# Filtering step 1: Remove low quality annotators

## Condition 1: Find annotators that only answered correctly to 0 or 1 control questions (c0-c5). 

In [2]:
# There are 5 extracts named c0, c1, c2, c3, and c4, their reference labels are in the control_questions dictionary.
# Use the data of annotations.csv, in lines of extract_id column equal to c0, c1, c2, c3, and c4, find anonymised_participant_id that the response is different from the respective reference of extract. 
# make a dictionary of anonymised_participant_id as key, and which extract (and number of extracts) the annotators answered wrong by annotator as value.

import pandas as pd

with open("./Annotations.csv", "r") as f:
     all_annotations = pd.read_csv(f)
    
control_questions = {'c0': 'Omstreden naar huidige maatstaven',
                     'c1': 'Omstreden naar huidige maatstaven',
                     'c2': 'Niet omstreden',
                     'c3': 'Omstreden naar huidige maatstaven',
                     'c4': 'Niet omstreden'}

# Filter the data to keep only the control questions
control_data = all_annotations[all_annotations['extract_id'].isin(control_questions.keys())]

# Create a dictionary to store the results
annotator_errors = {}

# Iterate over the control questions
for extract_id, reference_label in control_questions.items():
    # Get the rows for the current control question
    control_rows = control_data[control_data['extract_id'] == extract_id]
    
    # Find the annotators who answered incorrectly
    incorrect_annotators = control_rows[control_rows['response'] != reference_label]
    
    # Store the anonymised_participant_id and the number of incorrect answers
    for _, row in incorrect_annotators.iterrows():
        annotator_id = row['anonymised_participant_id']
        if annotator_id in annotator_errors:
            annotator_errors[annotator_id].append(extract_id)
        else:
            annotator_errors[annotator_id] = [extract_id]

# get the annotators that answered incorrectly to 3 to 5 control questions
wrong_annotators = {k: v for k, v in annotator_errors.items() if len(v) >= 3}

len(wrong_annotators)

30

## Condition 2: Mean (pairwise) alpha of the annotator < 0.2

In [4]:
# with the data from mean_alpha_per_annotator.csv, get a list of anonymised_participant_id where mean_alpha < 0.2

with open("./mean_alpha_per_annotator_original.csv", "r") as f:
     mean_alpha_per_annotator = pd.read_csv(f)

# Filter the data to keep only rows where mean_alpha < 0.2
filtered_data = mean_alpha_per_annotator[mean_alpha_per_annotator['mean_alpha'] < 0.2]

# Get the list of anonymised_participant_id where mean_alpha < 0.2
low_alpha_annotators = filtered_data['anonymised_participant_id'].tolist()

len(low_alpha_annotators)


83

## Filtering out annotators under above two conditions

In [5]:
# Remove rows where anonymised_participant_id is in the list of wrong_annotators and low_alpha_annotators
better_annotations = all_annotations[~all_annotations['anonymised_participant_id'].isin(wrong_annotators)]
better_annotations = better_annotations[~better_annotations['anonymised_participant_id'].isin(low_alpha_annotators)]

# Filtering step 2: remove the samples where response 'Onleesbare OCR' appeared. 

In [6]:
# Get extract_ids from data_annotations where response is 'Onleesbare OCR'
extract_ids_to_remove = all_annotations.loc[all_annotations['response'] == 'Onleesbare OCR', 'extract_id'].unique().tolist()

# Save control IDs
control_ids = ['c0', 'c1', 'c2', 'c3', 'c4']
for id in control_ids:
    extract_ids_to_remove.remove(id)

# Remove rows based on the extract_ids
annotations_no_OCR_issues = better_annotations[~better_annotations['extract_id'].isin(extract_ids_to_remove)]

# Filtering step 3: remove the annotations 'Onleesbare OCR' and 'Weet ik niet'.

In [8]:
# From Annotatons.csv, remove the rows where response is 'Onleesbare OCR' and 'Weet ik niet'. 
# For other rows, change the response from 'Omstreden naar huidige maatstaven' to 1 and 'Niet omstreden' to 0.
# remove rows where anonymised_participant_id in the list of wrong_annoataors and low_alpha_annotators

# Remove rows where response is 'Onleesbare OCR' or 'Weet ik niet'
final_annotations = annotations_no_OCR_issues[~annotations_no_OCR_issues['response'].isin(['Onleesbare OCR', 'Weet ik niet'])]

# Map 'Omstreden' to 1 and 'Niet omstreden' to 0
final_annotations['response'] = final_annotations['response'].map({'Omstreden naar huidige maatstaven': 1, 'Niet omstreden': 0})

# change 'response' to 'score'
final_annotations = final_annotations.rename(columns={'response': 'score'})

# Save the filtered data to a new CSV file
final_annotations.to_csv("./ready_annotations.csv", index=False)

# Display or save the filtered data
final_annotations


/var/folders/5w/511b227d2z53cxgmdw8rgp400000gp/T/ipykernel_76286/1207291337.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_annotations['response'] = final_annotations['response'].map({'Omstreden naar huidige maatstaven': 1, 'Niet omstreden': 0})


,anonymised_participant_id,extract_id,score,suggestion
0,unknown_2a,H99,0,NaN
1,unknown_2b,H99,0,NaN
2,unknown_2c,H99,0,NaN
3,unknown_2d,H99,0,NaN
4,unknown_2e,H99,0,NaN
...,...,...,...,...
21780,1,2,1,NaN
21781,2,2,1,negerhut
21783,4,2,1,Negerhut
21784,5,2,1,NaN
